In [11]:
import os
import contextlib
import logging
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path


In [12]:
# ==================== CẤU HÌNH & HẰNG SỐ ====================
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

DIRTY_STOCK_DIR = DATA_DIR / "dirty" / "stocks"
DIRTY_ETF_DIR = DATA_DIR / "dirty" / "etfs"

QUARANTINE_STOCK_DIR = DATA_DIR / "quarantine" / "stocks"
QUARANTINE_ETF_DIR = DATA_DIR / "quarantine" / "etfs"

METADATA_DIR = DATA_DIR / "metadata"
REPORT_DIR = DATA_DIR / "reports"

DIRECTORIES = [
    DIRTY_STOCK_DIR,
    DIRTY_ETF_DIR,
    QUARANTINE_STOCK_DIR,
    QUARANTINE_ETF_DIR,
    METADATA_DIR,
    REPORT_DIR,
]
              
NASDAQ_URL = "http://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt"

In [13]:
# ==================== CÁC HÀM XỬ LÝ ====================
def setup_directories(dirs: list):
    for d in dirs:
        os.makedirs(d, exist_ok=True)
    logging.info("Đã thiết lập xong cấu trúc thư mục.")

def strict_validate_stock_data(df: pd.DataFrame, symbol: str) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Kiểm tra chất lượng dữ liệu tài chính.
    Tách luồng dữ liệu thành Sạch (Clean) và Cách ly (Quarantine).
    """
    df = df.copy()
    original_rows = len(df)
    
    # 1. Ép kiểu Datetime64 cho Date
    try:
        df['Date'] = pd.to_datetime(df['Date'])
    except Exception as e:
        raise ValueError(f"[{symbol}] Cột Date không đúng định dạng Datetime64. Lỗi: {e}")

    # 2. Xử lý Trùng lặp & Tăng dần
    dup_mask = df.duplicated(subset=['Date'], keep='first')
    total_duplicates = dup_mask.sum()
    df = df[~dup_mask].sort_values('Date').reset_index(drop=True)

    # 3. Auto-Fix High/Low
    df['High'] = df[['High', 'Open', 'Close']].max(axis=1)
    df['Low'] = df[['Low', 'Open', 'Close']].min(axis=1)

    # 4. Gán Tag lỗi (Quarantine Validation)
    price_cols = ["Open", "High", "Low", "Close"]
    if "Adj Close" in df.columns:
        price_cols.append("Adj Close")
        
    df['invalid_reason'] = ""
    
    # Logic phát hiện lỗi
    negative_price_mask = (df[price_cols] <= 0).any(axis=1)
    df.loc[negative_price_mask, 'invalid_reason'] += "NEGATIVE_OR_ZERO_PRICE;"
    
    negative_volume_mask = df['Volume'] < 0
    df.loc[negative_volume_mask, 'invalid_reason'] += "NEGATIVE_VOLUME;"
    
    nan_mask = df[price_cols + ['Volume']].isna().any(axis=1)
    df.loc[nan_mask, 'invalid_reason'] += "MISSING_VALUE;"
    
    # 5. Phân tách Dữ liệu
    is_invalid = df['invalid_reason'] != ""
    quarantine_df = df[is_invalid].copy()
    clean_df = df[~is_invalid].copy()
    
    # Sắp xếp lại cột cho Quarantine
    if not quarantine_df.empty:
        quarantine_df['Symbol'] = symbol
        cols_order = ['Date', 'Symbol', 'invalid_reason'] + [c for c in quarantine_df.columns if c not in ['Date', 'Symbol', 'invalid_reason']]
        quarantine_df = quarantine_df[cols_order]

    # Dọn dẹp Clean DataFrame
    clean_df = clean_df.drop(columns=['invalid_reason'])
    clean_df['Volume'] = clean_df['Volume'].fillna(0).astype('int64')

    # 6. Gom Metrics
    metrics = {
        "Symbol": symbol,
        "Original": original_rows,
        "Duplicate_Dropped": total_duplicates,
        "Negative_Price_Rows": int(negative_price_mask.sum()),
        "Negative_Volume_Rows": int(negative_volume_mask.sum()),
        "Missing_Value_Rows": int(nan_mask.sum()),
        "Total_Quarantined": len(quarantine_df),
        "Remain": len(clean_df)
    }
    
    return clean_df, quarantine_df, metrics

def fetch_and_clean_metadata() -> pd.DataFrame:
    logging.info("Đang tải danh sách symbols từ NASDAQ...")
    dirty_data = pd.read_csv(NASDAQ_URL, sep='|')
    clean_data = dirty_data[dirty_data['Test Issue'] == 'N'].copy()
    clean_data = clean_data[~clean_data['NASDAQ Symbol'].str.contains(r'[-=\.\^]', regex=True, na=False)]
    return clean_data


In [14]:
# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    setup_directories(DIRECTORIES)
    meta_df = fetch_and_clean_metadata()
    symbols = meta_df['NASDAQ Symbol'].tolist()
    symbol_type_map = dict(zip(meta_df['NASDAQ Symbol'], meta_df['ETF']))

    valid_symbols = []
    all_metrics = [] # Danh sách lưu log report

    logging.info("Bắt đầu tải dữ liệu và phân luồng Clean/Quarantine...")
    
    with open(os.devnull, 'w') as devnull:
        with contextlib.redirect_stdout(devnull):
            for symbol in symbols:
                try:
                    df = yf.download(symbol, period='max', progress=False)
                    if df.empty: continue
                    
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = df.columns.droplevel(1)
                    
                    df = df.reset_index()
                    if df['Date'].dt.tz is not None:
                        df['Date'] = df['Date'].dt.tz_localize(None)
                    df['Symbol'] = symbol

                    # Chạy hàm Validate mới
                    clean_df, quarantine_df, metrics = strict_validate_stock_data(df, symbol)
                    all_metrics.append(metrics)

                    is_etf = symbol_type_map.get(symbol) == 'Y'
                    
                    # Lưu file clean
                    if not clean_df.empty:
                        dirty_folder = DIRTY_ETF_DIR if is_etf else DIRTY_STOCK_DIR
                        clean_df.to_parquet(f"{dirty_folder}/{symbol}.parquet", index=False, engine='pyarrow', compression='snappy')
                        valid_symbols.append(symbol)
                    
                    # Lưu file quarantine
                    if not quarantine_df.empty:
                        q_folder = QUARANTINE_ETF_DIR if is_etf else QUARANTINE_STOCK_DIR
                        quarantine_df.to_parquet(f"{q_folder}/{symbol}_error.parquet", index=False, engine='pyarrow', compression='snappy')
                        
                except Exception as e:
                    logging.warning(f"Lỗi khi xử lý {symbol}: {e}")
                    continue

    # Xuất Report thống kê tổng
    report_df = pd.DataFrame(all_metrics)
    report_df.to_csv(REPORT_DIR/"symbol_quality_report_1_data_colect.csv", index=False)
    logging.info("Hoàn thành! Đã xuất symbol_quality_report.csv")

    valid_meta = meta_df[meta_df['NASDAQ Symbol'].isin(valid_symbols)]
    valid_meta[valid_meta['ETF'] == 'N'].to_csv(METADATA_DIR / "symbols_stock.csv", index=False)
    valid_meta[valid_meta['ETF'] == 'Y'].to_csv(METADATA_DIR / "symbols_etf.csv", index=False)

INFO: Đã thiết lập xong cấu trúc thư mục.
INFO: Đang tải danh sách symbols từ NASDAQ...
INFO: Bắt đầu tải dữ liệu và phân luồng Clean/Quarantine...
ERROR: AACBR: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: 
1 Failed download:
ERROR: ['AACBR']: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: AACOW: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: 
1 Failed download:
ERROR: ['AACOW']: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: AACPW: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: 
1 Failed download:
ERROR: ['AACPW']: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: AAPE: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: 
1 Failed download:
ERROR: ['AAPE']: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: ACAAW: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: 
1 Failed download:
ERROR: ['ACAAW']: Period 'max' is invalid, must be one of: 1d, 5d
ERROR: $ACHR+: possibly delisted; no timezone found
ERROR: 
1 Failed download:
